In [ ]:
# ============================================================
# 1) valid pid 추출
# ============================================================
def get_valid_pids_with_required_vars(
    feather_path: str,
    required_itemids=(1, 2, 3, 4, 10),
):
    df = pd.read_feather(feather_path)

    counts = (
        df.groupby(["pid", "itemid"])
          .size()
          .unstack(fill_value=0)
    )

    required_itemids = list(required_itemids)
    for c in required_itemids:
        if c not in counts.columns:
            counts[c] = 0

    valid_pids = (
        counts.loc[(counts[required_itemids] > 0).all(axis=1)]
              .index
              .tolist()
    )
    return valid_pids


# ============================================================
# 2) 모델 1개를 한 번만 준비
# ============================================================
def prepare_variant_context(
    *,
    model_name: str,
    source_domain: str = "mimic",
    source_split_id: int = 1,
    target_domain: str | None = None,
    fixed_target_test_split: int = 1,
    device: str | None = None,
    data_root: str = "./data",
    output_directory: str = "./surprise_results",
    project_name: str = "CV_5fold",
    model_cfg_json: str = "{}",
    var_names: dict[int, str] | None = None,
    plot_only_var_names: list[str] | None = None,
    auto_inverse_values_from_stats: bool = True,
    inverse_time_fn=None,
    inverse_value_map=None,
    compute_loader_mask_summary: bool = True,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if target_domain is None:
        target_domain = "eicu" if source_domain == "mimic" else "mimic"

    split_out = os.path.join(output_directory, f"{project_name}_split{source_split_id}")
    fig_dir = os.path.join(split_out, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    # outcome cols
    ocs_path = os.path.join(data_root, "split1", f"{source_domain}_outcomes_train_1.feather")
    ocs_df = pd.read_feather(ocs_path)
    ocs = ocs_df.columns.tolist()
    if "pid" in ocs:
        ocs.remove("pid")

    args = make_default_args(
        model_name=model_name,
        source_domain=source_domain,
        target_domain=target_domain,
        data_root=data_root,
        output_directory=output_directory,
        project_name=project_name,
        model_cfg_json=model_cfg_json,
        fixed_target_test_split=fixed_target_test_split,
    )
    cfg = build_feather_cfg_from_args(args, ocs)
    if not hasattr(cfg, "model_cfg") or cfg.model_cfg is None:
        cfg.model_cfg = json.loads(model_cfg_json)

    best_ckpt = find_best_ckpt(
        output_directory=cfg.output_directory,
        project_name=cfg.project_name,
        split_id=source_split_id,
        stage="train",
        monitor="val_AUPRC_macro",
        mode="max",
        model_name=model_name,
        source_domain=source_domain,
    )

    vis_bundle = build_loaders_for_vis(cfg, split_id=int(source_split_id), return_raw=True)

    num_features = vis_bundle["num_features"]
    ld_te = vis_bundle["source_test_loader"]
    target_loader = vis_bundle["target_test_loader"]
    stats = vis_bundle["stats"]

    if inverse_time_fn is None:
        inverse_time_fn = inverse_time_default
    if inverse_value_map is None and auto_inverse_values_from_stats:
        inverse_value_map = make_inverse_value_map_from_stats(stats)

    plot_only_vars = None
    if plot_only_var_names is not None:
        if var_names is None:
            raise ValueError("plot_only_var_names requires var_names.")
        inv_var_names = invert_var_names(var_names)
        missing = [n for n in plot_only_var_names if n not in inv_var_names]
        if missing:
            raise ValueError(f"Unknown variable names in plot_only_var_names: {missing}")
        plot_only_vars = [inv_var_names[n] for n in plot_only_var_names]

    # restore model once
    n_output = getattr(cfg, "n_output", 1)
    core_model = build_model(
        cfg.model_name,
        cfg.model_cfg,
        num_features=num_features,
        n_output=n_output,
    )

    lit = STRaTSLit(
        model=core_model,
        task_type=cfg.task_type,
        outcome_cols=cfg.outcome_cols,
        target_indices=cfg.target_indices,
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        optimizer=cfg.optimizer,
    )

    ckpt = torch.load(best_ckpt, map_location="cpu")
    lit.load_state_dict(ckpt["state_dict"], strict=False)
    lit.eval().to(device)
    net = lit.model.eval()

    src_mask_count_df = None
    src_mask_overall = None
    tgt_mask_count_df = None
    tgt_mask_overall = None

    if compute_loader_mask_summary and model_name != "strats":
        src_mask_count_df, src_mask_overall = collect_mask_counts_over_loader(
            net, ld_te, device=device, var_names=var_names
        )
        tgt_mask_count_df, tgt_mask_overall = collect_mask_counts_over_loader(
            net, target_loader, device=device, var_names=var_names
        )

    return {
        "model_name": model_name,
        "cfg": cfg,
        "device": device,
        "best_ckpt": best_ckpt,
        "fig_dir": fig_dir,
        "net": net,
        "ld_te": ld_te,
        "target_loader": target_loader,
        "stats": stats,
        "inverse_time_fn": inverse_time_fn,
        "inverse_value_map": inverse_value_map,
        "plot_only_vars": plot_only_vars,
        "var_names": var_names,
        "source_domain": source_domain,
        "target_domain": target_domain,
        "src_mask_count_df": src_mask_count_df,
        "src_mask_overall": src_mask_overall,
        "tgt_mask_count_df": tgt_mask_count_df,
        "tgt_mask_overall": tgt_mask_overall,
    }


# ============================================================
# 3) context에서 pid pair 하나만 처리
# ============================================================
def run_pair_from_context(
    ctx,
    *,
    pid_source: int,
    pid_target: int,
    save_plots: bool = True,
):
    model_name = ctx["model_name"]
    net = ctx["net"]
    ld_te = ctx["ld_te"]
    target_loader = ctx["target_loader"]
    device = ctx["device"]
    inverse_time_fn = ctx["inverse_time_fn"]
    inverse_value_map = ctx["inverse_value_map"]
    plot_only_vars = ctx["plot_only_vars"]
    var_names = ctx["var_names"]
    source_domain = ctx["source_domain"]
    target_domain = ctx["target_domain"]
    fig_dir = ctx["fig_dir"]

    if model_name == "strats":
        raise ValueError("strats has no learned masking for this visualization.")

    inst_src = find_instance_by_pid(ld_te, pid_source)
    if inst_src is None:
        raise ValueError(f"[SOURCE] pid={pid_source} not found in source test loader.")

    inst_tgt = find_instance_by_pid(target_loader, pid_target)
    if inst_tgt is None:
        raise ValueError(f"[TARGET] pid={pid_target} not found in target test loader.")

    src_dbg = run_check_padding(net, *inst_src, device=device)
    tgt_dbg = run_check_padding(net, *inst_tgt, device=device)

    src_df_pid = pid_tensors_to_long_df(
        pid=pid_source,
        times1=src_dbg["times"],
        varis1=src_dbg["varis"],
        values1=src_dbg["values"],
        pad1=inst_src[3],
        redundant_mask1=src_dbg["mask"],
        inverse_time_fn=inverse_time_fn,
        inverse_value_map=inverse_value_map,
    )

    tgt_df_pid = pid_tensors_to_long_df(
        pid=pid_target,
        times1=tgt_dbg["times"],
        varis1=tgt_dbg["varis"],
        values1=tgt_dbg["values"],
        pad1=inst_tgt[3],
        redundant_mask1=tgt_dbg["mask"],
        inverse_time_fn=inverse_time_fn,
        inverse_value_map=inverse_value_map,
    )

    model_name_formatted = format_model_name(model_name)
    source_pretty = format_source_name(source_domain)
    target_pretty = format_source_name(target_domain)

    src_title = f"Masks from {model_name_formatted} | Source={source_pretty} | Data={source_pretty}"
    tgt_title = f"Masks from {model_name_formatted} | Source={source_pretty} | Data={target_pretty}"

    src_recon_path = None
    tgt_recon_path = None
    if save_plots:
        src_recon_path = os.path.join(
            fig_dir,
            f"reconstruction_source_{model_name}_{source_domain}_pid{pid_source}.png",
        )
        tgt_recon_path = os.path.join(
            fig_dir,
            f"reconstruction_target_{model_name}_{target_domain}_pid{pid_target}.png",
        )

    src_var_mse_df, src_overall, src_pid_var_detail = masked_reconstruction_mse(
        src_df_pid,
        var_names=var_names,
        pid_filter=pid_source,
        pid_target=pid_source,
        plot_only_vars=plot_only_vars,
        time_col="times",
        var_col="varis",
        value_col="values",
        mask_col="mask",
        time_unit="min",
        save_path=src_recon_path,
        suptitle=src_title,
        fixed_var_ylims=FIXED_VAR_YLIMS,
    )

    tgt_var_mse_df, tgt_overall, tgt_pid_var_detail = masked_reconstruction_mse(
        tgt_df_pid,
        var_names=var_names,
        pid_filter=pid_target,
        pid_target=pid_target,
        plot_only_vars=plot_only_vars,
        time_col="times",
        var_col="varis",
        value_col="values",
        mask_col="mask",
        time_unit="min",
        save_path=tgt_recon_path,
        suptitle=tgt_title,
        fixed_var_ylims=FIXED_VAR_YLIMS,
    )

    return {
        "model_name": model_name,
        "pid_source": pid_source,
        "pid_target": pid_target,
        "src_df_pid": src_df_pid,
        "tgt_df_pid": tgt_df_pid,
        "src_check_padding": src_dbg,
        "tgt_check_padding": tgt_dbg,
        "src_var_mse_df": src_var_mse_df,
        "src_overall_mse": src_overall,
        "src_pid_var_detail": src_pid_var_detail,
        "tgt_var_mse_df": tgt_var_mse_df,
        "tgt_overall_mse": tgt_overall,
        "tgt_pid_var_detail": tgt_pid_var_detail,
        "src_mask_overall": ctx["src_mask_overall"],
        "tgt_mask_overall": ctx["tgt_mask_overall"],
        "src_recon_path": src_recon_path,
        "tgt_recon_path": tgt_recon_path,
        "figure_dir": ctx["fig_dir"],
        "cfg": ctx["cfg"],
    }


# ============================================================
# 4) 여러 pid pair를 효율적으로 순회
# ============================================================
def run_all_pairs_efficient(
    *,
    source_domain="mimic",
    target_domain="eicu",
    source_split_id=1,
    data_root="./data",
    output_directory="./surprise_results",
    project_name="CV_5fold",
    model_cfg_json="{}",
    var_names=None,
    selected_vars=None,
    model_names=("surprise", "surprise_vt", "surprise_vttg"),
    required_itemids=(1, 2, 3, 4, 10),
    save_overlays=True,
):
    # valid pids once
    pid_m = get_valid_pids_with_required_vars(
        os.path.join(data_root, "split1", "mimic_data_test_1.feather"),
        required_itemids=required_itemids,
    )
    pid_e = get_valid_pids_with_required_vars(
        os.path.join(data_root, "split1", "eicu_data_test_1.feather"),
        required_itemids=required_itemids,
    )

    n_pairs = min(len(pid_m), len(pid_e))
    print(f"valid MIMIC pids: {len(pid_m)}")
    print(f"valid eICU pids: {len(pid_e)}")
    print(f"paired runs: {n_pairs}")

    # prepare each model once
    contexts = {}
    for model_name in model_names:
        print(f"\nPreparing context: {model_name}")
        contexts[model_name] = prepare_variant_context(
            model_name=model_name,
            source_domain=source_domain,
            source_split_id=source_split_id,
            target_domain=target_domain,
            data_root=data_root,
            output_directory=output_directory,
            project_name=project_name,
            model_cfg_json=model_cfg_json,
            var_names=var_names,
            plot_only_var_names=selected_vars,
            compute_loader_mask_summary=True,
        )

        print(f"[{model_name}] source loader mask summary:", contexts[model_name]["src_mask_overall"])
        print(f"[{model_name}] target loader mask summary:", contexts[model_name]["tgt_mask_overall"])

    all_results = []

    pretty_name_map = {
        "surprise": "Surprise",
        "surprise_vt": "SurpVT",
        "surprise_vttg": "SurpVTTG",
        "strats": "STraTS",
    }

    for i in range(n_pairs):
        pid_source = int(pid_m[i])
        pid_target = int(pid_e[i])

        print("\n" + "=" * 100)
        print(f"[{i+1}/{n_pairs}] pid_source={pid_source}, pid_target={pid_target}")

        pair_results = {}

        for model_name in model_names:
            res = run_pair_from_context(
                contexts[model_name],
                pid_source=pid_source,
                pid_target=pid_target,
                save_plots=True,
            )
            pair_results[pretty_name_map.get(model_name, model_name)] = res

            print(f"[{model_name}] source overall:", res["src_mask_overall"])
            print(f"[{model_name}] target overall:", res["tgt_mask_overall"])

        overlay_out = None
        if save_overlays:
            overlay_out = make_source_target_variant_overlays(
                pair_results,
                var_names=var_names,
                plot_only_var_names=selected_vars,
                source_overlay_filename=f"overlay_variants_source_{pid_source}.png",
                target_overlay_filename=f"overlay_variants_target_{pid_target}.png",
            )

        all_results.append({
            "i": i,
            "pid_source": pid_source,
            "pid_target": pid_target,
            "pair_results": pair_results,
            "overlay_out": overlay_out,
        })

    return {
        "pid_m": pid_m,
        "pid_e": pid_e,
        "contexts": contexts,
        "all_results": all_results,
    }

In [ ]:
selected_vars = ["heart_rate", "map", "resp_rate", "temperature", "glucose"]

out = run_all_pairs_efficient(
    source_domain="mimic",
    target_domain="eicu",
    source_split_id=1,
    data_root="./data",
    output_directory="./surprise_results",
    project_name="CV_5fold",
    var_names=var_names,
    selected_vars=selected_vars,
    model_names=("surprise", "surprise_vt", "surprise_vttg"),
    required_itemids=(1, 2, 3, 4, 10),
)

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
# plt.rcParams['font.size'] = 22

#pid_source = 21206625
#pid_target = 2210186
# pid_target = 3239318
# pid_source = 20263113 surpvt가 다 가림
# pid_target = 579301
# pid_target = 613584 surpvt가 거의 못 따라

# 얘네는 surpvt만이 문제...
# pid_source = 20266641
# pid_target = 621660


# pid_source = 20271349
# pid_target = 637979

pid_source = 20311746
#pid_target = 692495 일단 보류
pid_target = 754429

# res1 = run_split1_pipeline(
#     model_name="strats",
#     source_domain="mimic",
#     source_split_id=1,
#     pid_source=pid_source,
#     pid_target=pid_target,
#     do_pacmap=False,
#     do_masked_recon_plot=False,
#     var_names=var_names,
#     plot_only_var_names=selected_vars
# )

# print(res1['src_mask_overall'])
# print(res1['tgt_mask_overall'])

res2 = run_split1_pipeline(
    model_name="surprise",
    source_domain="mimic",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=False,
    do_masked_recon_plot=True,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res2['src_mask_overall'])
print(res2['tgt_mask_overall'])

res3 = run_split1_pipeline(
    model_name="surprise_vt",
    source_domain="mimic",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=False,
    do_masked_recon_plot=True,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res3['src_mask_overall'])
print(res3['tgt_mask_overall'])

res4 = run_split1_pipeline(
    model_name="surprise_vttg",
    source_domain="mimic",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=False,
    do_masked_recon_plot=True,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res4['src_mask_overall'])
print(res4['tgt_mask_overall'])

overlay_out = make_source_target_variant_overlays(
    {
        "Surprise": res2,
        "SurpVT": res3,
        "SurpVTTG": res4,
    },
    var_names=var_names,
    plot_only_var_names=selected_vars,
    source_overlay_filename=f"overlay_variants_source_{pid_source}.png",
    target_overlay_filename=f"overlay_variants_target_{pid_target}.png",
)

In [ ]:
overlay_out = make_source_target_variant_overlays(
    {
        "Surprise": res2,
        "SurpVT": res3,
        "SurpVTTG": res4,
    },
    var_names=var_names,
    plot_only_var_names=selected_vars,
)

In [ ]:
pid_source = 637979
pid_target = 20271349


res5 = run_split1_pipeline(
    model_name="strats",
    source_domain="eicu",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=True,
    do_masked_recon_plot=False,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res5['src_mask_overall'])
print(res5['tgt_mask_overall'])

res6 = run_split1_pipeline(
    model_name="surprise",
    source_domain="eicu",
    source_split_id=1,
    pid_source=pid_source,
    pid_target=pid_target,
    do_pacmap=True,
    do_masked_recon_plot=False,
    var_names=var_names,
    plot_only_var_names=selected_vars
)

print(res6['src_mask_overall'])
print(res6['tgt_mask_overall'])

# res7 = run_split1_pipeline(
#     model_name="surprise_vt",
#     source_domain="eicu",
#     source_split_id=1,
#     pid_source=pid_source,
#     pid_target=pid_target,
#     do_pacmap=True,
#     do_masked_recon_plot=True,
#     var_names=var_names,
#     plot_only_var_names=selected_vars
# )

# print(res7['src_mask_overall'])
# print(res7['tgt_mask_overall'])

# res8 = run_split1_pipeline(
#     model_name="surprise_vttg",
#     source_domain="eicu",
#     source_split_id=1,
#     pid_source=pid_source,
#     pid_target=pid_target,
#     do_pacmap=True,
#     do_masked_recon_plot=True,
#     var_names=var_names,
#     plot_only_var_names=selected_vars
# )

# print(res8['src_mask_overall'])
# print(res8['tgt_mask_overall'])
